# Session: Audio-Visual Data Archiving

**Thursday Masking Day — Tilburg Multiscale Summerschool 2026**

After you mask your videos you need to archive them in a way that:
1. Preserves a clear link between **raw**, **masked**, and **derived** files  
2. Records exactly **which masking parameters** were used (for reproducibility)  
3. Packages the data in a format that **repositories and ethics boards** accept  
4. Supports future researchers in re-using or re-masking the data

This notebook walks through a lightweight but complete archiving workflow grounded in the Edelman/Envision dataset.

---
### Why archiving matters for masked video

Unlike questionnaire data, masked video has a **dual provenance problem**:

| Question | Answer needed for archive |
|----------|---------------------------|
| What was the original? | Raw four-up composite + camera setup metadata |
| What was masked? | Which streams, which masking strategy, which model version |
| What was preserved? | Pose time series, audio track, timecodes |
| Under what consent? | IRB protocol number, consent scope |
| Who can access what? | Raw = restricted; masked = open or embargoed |

**Reference:** Owoyele et al. (2026). MaskingOPS: A Tutorial for Masking Operations in Behavioral Research. *BRM* (under review).


## Setup

In [ ]:
import json
import hashlib
import subprocess
import csv
import shutil
from pathlib import Path
from datetime import datetime, timezone

REPO_ROOT    = Path("../")
ARCHIVE_ROOT = REPO_ROOT / "data" / "archived"
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Metadata constants — fill in for your study
STUDY_METADATA = {
    "study_name":       "Envision Triadic Design Study",
    "original_authors": "Edelman, J.A. (2011)",
    "reuse_context":    "Tilburg Multiscale Summerschool 2026",
    "irb_protocol":     "INSERT_IRB_NUMBER",
    "consent_scope":    "anonymized research / non-commercial educational use",
    "masking_tool":     "MaskAnyone v1.x (SYNAPSIS, Radboud University)",
    "masking_method":   "YOLO + SAM2 instance segmentation, body silhouette",
    "archived_by":      "Owoyele, B.",
}

## Part 1: File naming convention

Consistent naming is the cheapest form of provenance.  
We use the pattern:

```
{study}__{group}__{stream}__{version}__{status}.mp4
```

| Field | Example values |
|-------|---------------|
| `study` | `envision` |
| `group` | `group_02`, `group_07` |
| `stream` | `cam_p1`, `cam_p2`, `cam_p3`, `cam_overhead` |
| `version` | `v01`, `v02` (masking iteration) |
| `status` | `raw`, `masked`, `clip` |

In [ ]:
def make_archive_name(study: str, group: str, stream: str,
                       version: str, status: str, ext: str = ".mp4") -> str:
    return f"{study}__{group}__{stream}__{version}__{status}{ext}"

# Examples
for stream in ["cam_p1", "cam_p2", "cam_p3"]:
    print(make_archive_name("envision", "group_07", stream, "v01", "masked"))

## Part 2: Compute checksums

A SHA-256 checksum ties each archived file to its exact byte content.  
If a file is corrupted or accidentally re-encoded, the checksum will not match.

In [ ]:
def sha256(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()

def probe_video(path: Path) -> dict:
    cmd = ["ffprobe", "-v", "quiet", "-print_format", "json",
           "-show_streams", "-show_format", str(path)]
    return json.loads(subprocess.run(cmd, capture_output=True, text=True).stdout)

# Quick demo on the sample clip
sample = REPO_ROOT / "__design.mp4"
print("SHA-256:", sha256(sample))

## Part 3: Per-file sidecar metadata (JSON)

Every archived video gets a `.json` sidecar that records everything a future  
researcher needs to understand and re-use it.

In [ ]:
def make_sidecar(video_path: Path, archive_name: str,
                  group: str, stream: str, status: str,
                  masking_params: dict = None,
                  notes: str = "") -> dict:
    meta  = probe_video(video_path)
    fmt   = meta["format"]
    vid   = next((s for s in meta["streams"] if s["codec_type"] == "video"), {})
    aud   = next((s for s in meta["streams"] if s["codec_type"] == "audio"), None)

    return {
        "archive_name":    archive_name,
        "sha256":          sha256(video_path),
        "archived_at":     datetime.now(timezone.utc).isoformat(),
        "study":           STUDY_METADATA,
        "provenance": {
            "group":       group,
            "stream":      stream,
            "status":      status,   # "raw" | "masked" | "clip"
            "source_file": video_path.name,
        },
        "video": {
            "codec":      vid.get("codec_name"),
            "width":      vid.get("width"),
            "height":     vid.get("height"),
            "fps":        vid.get("r_frame_rate"),
            "duration_s": float(fmt.get("duration", 0)),
            "size_bytes": int(fmt.get("size", 0)),
        },
        "audio": {
            "present":     aud is not None,
            "codec":       aud.get("codec_name") if aud else None,
            "sample_rate": aud.get("sample_rate") if aud else None,
        },
        "masking_params":  masking_params or {},
        "notes":           notes,
    }

# Demo sidecar for the sample clip
sidecar = make_sidecar(
    video_path    = sample,
    archive_name  = make_archive_name("envision", "sample", "composite", "v00", "raw"),
    group         = "sample",
    stream        = "composite",
    status        = "raw",
    notes         = "20-second reference clip bundled with summerschool repo"
)
print(json.dumps(sidecar, indent=2))

## Part 4: Archive a batch of masked videos

In [ ]:
def archive_batch(source_dir: Path, archive_root: Path,
                   study: str, group: str, version: str, status: str,
                   masking_params: dict = None) -> list:
    """
    Copy all .mp4 files from source_dir into archive_root with canonical names,
    writing a .json sidecar alongside each file.

    Returns list of archive record dicts (suitable for writing to a manifest CSV).
    """
    group_archive = archive_root / group
    group_archive.mkdir(parents=True, exist_ok=True)

    records = []
    for src in sorted(source_dir.glob("*.mp4")):
        # infer stream from filename (expects __cam_XX__ segment)
        parts  = src.stem.split("__")
        stream = next((p for p in parts if p.startswith("cam_")), "unknown")

        arc_name = make_archive_name(study, group, stream, version, status)
        dst_video = group_archive / arc_name
        dst_json  = group_archive / arc_name.replace(".mp4", ".json")

        shutil.copy2(src, dst_video)

        sidecar = make_sidecar(src, arc_name, group, stream, status, masking_params)
        dst_json.write_text(json.dumps(sidecar, indent=2))

        records.append({"archive_name": arc_name, "group": group,
                         "stream": stream, "sha256": sidecar["sha256"],
                         "duration_s": sidecar["video"]["duration_s"],
                         "size_mb": round(sidecar["video"]["size_bytes"] / 1e6, 2)})
        print(f"  archived → {arc_name}")

    return records

# Example masking parameters to record
MASKING_PARAMS_EXAMPLE = {
    "tool":          "MaskAnyone",
    "detection":     "YOLOv8x",
    "segmentation":  "SAM2-large",
    "strategy":      "body_silhouette_black",
    "confidence":    0.75,
    "pose_overlay":  False,
    "server":        "SYNAPSIS Ponyland",
    "date_processed":datetime.now(timezone.utc).date().isoformat(),
}

print("archive_batch() ready — run after your MaskAnyone outputs are downloaded.")

## Part 5: Master archive manifest

Combine all per-batch records into one `ARCHIVE_MANIFEST.csv` at the archive root.

In [ ]:
def write_master_manifest(archive_root: Path):
    rows = []
    for json_file in sorted(archive_root.rglob("*.json")):
        meta = json.loads(json_file.read_text())
        rows.append({
            "archive_name":  meta["archive_name"],
            "group":         meta["provenance"]["group"],
            "stream":        meta["provenance"]["stream"],
            "status":        meta["provenance"]["status"],
            "duration_s":    meta["video"]["duration_s"],
            "size_mb":       round(meta["video"]["size_bytes"] / 1e6, 2),
            "codec":         meta["video"]["codec"],
            "sha256":        meta["sha256"],
            "archived_at":   meta["archived_at"],
            "masking_tool":  meta["masking_params"].get("tool", ""),
            "masking_method":meta["masking_params"].get("strategy", ""),
            "notes":         meta["notes"],
        })

    if not rows:
        print("No archived files found yet.")
        return

    manifest_path = archive_root / "ARCHIVE_MANIFEST.csv"
    with open(manifest_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    print(f"Manifest: {manifest_path}  ({len(rows)} files)")
    return manifest_path

write_master_manifest(ARCHIVE_ROOT)

## Part 6: Access tiers

Masked video still requires **tiered access** even after anonymisation — faces may  
not be the only re-identification risk (voice, workspace environment, sketches).

| Tier | Content | Access |
|------|---------|--------|
| **Restricted** | Raw four-up composites | Named researchers only, ethics board approval |
| **Embargoed** | Masked videos (pre-publication) | Collaborators only until paper is accepted |
| **Open** | Masked clips + manifest + sidecar JSONs | CC-BY-NC (aligned with Edelman 2011 license) |
| **Derived** | Pose time series CSVs, transcripts | Open, no video content |

For Radboud University: use the **[RDR — Radboud Data Repository](https://data.ru.nl)**  
with DANS/EASY metadata export for FAIR compliance.

For the summerschool: keep raw files on **SYNAPSIS** (not in this repo) and share  
only the masked clips and manifest through the workshop GitHub.


## Part 7: Extract and archive audio separately

Audio is often more privacy-sensitive than video (voice is biometric).  
We extract it as a separate track so access can be controlled independently.

In [ ]:
def extract_audio(video_path: Path, out_dir: Path, fmt: str = "wav") -> Path:
    """Extract audio track from video to wav/mp3/aac."""
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / (video_path.stem + f".{fmt}")
    cmd = [
        "ffmpeg", "-y",
        "-i",  str(video_path),
        "-vn",
        "-acodec", "pcm_s16le" if fmt == "wav" else fmt,
        str(out_path)
    ]
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode != 0:
        print(f"  No audio track in {video_path.name} — skipping.")
        return None
    return out_path

def strip_audio(video_path: Path, out_path: Path) -> Path:
    """Create a video-only copy (no audio) for open-tier distribution."""
    cmd = [
        "ffmpeg", "-y",
        "-i",  str(video_path),
        "-an",
        "-c:v", "copy",
        str(out_path)
    ]
    subprocess.run(cmd, capture_output=True, check=True)
    return out_path

print("Audio extraction helpers ready.")
print("Connection to Tuesday (acoustic analysis):")
print("  extract_audio() → .wav → Tuesday pitch/intensity pipeline")
print("  strip_audio()   → silent masked video for open-tier archive")

## Part 8: DANS and open data infrastructure (Netherlands)

The Dutch national research data infrastructure gives behavioral AV data a clear  
path from your hard drive to a citeable, FAIR-compliant open archive.

### Key repositories

| Repository | URL | Best for |
|------------|-----|----------|
| **DANS Data Station SSH** | ssh.datastations.nl | Social science AV data (primary target) |
| **Radboud Data Repository (RDR)** | data.ru.nl | Radboud-affiliated studies (pre-deposit staging) |
| **DataverseNL** | dataverse.nl | National Dataverse, institutional collections |
| **CLARIN ERIC / DANS** | clarin.eu | Language + multimodal behavioral corpora |
| **4TU.ResearchData** | data.4tu.nl | Engineering / technical datasets |

### Metadata standards expected by DANS

| Standard | Purpose |
|----------|---------|
| **DataCite 4.x** | DOI registration (title, creators, license, dates) |
| **Dublin Core** | Basic discovery metadata |
| **DDI Codebook** | Social science variables and instruments |
| **CMDI** | Component Metadata for multimodal/linguistic corpora |

For masked video: **DataCite** + a **README.txt** describing the masking procedure  
is sufficient for DANS Data Station SSH.

DANS uses FAIR principles (Findable, Accessible, Interoperable, Reusable).  
Our sidecar JSON + checksums + canonical file names are already aligned with this.

In [ ]:
def make_datacite_metadata(sidecar: dict, doi_placeholder: str = "10.XXXXX/envision") -> dict:
    """
    Generate DataCite 4.x compatible metadata dict from a sidecar JSON.
    Upload this as datacite.json alongside your DANS deposit.
    """
    study = sidecar["study"]
    return {
        "@context": "http://schema.org",
        "@type":    "Dataset",
        "identifier": doi_placeholder,
        "name": (
            f"{study['study_name']} — "
            f"{sidecar['provenance']['group']} "
            f"({sidecar['provenance']['stream']}, {sidecar['provenance']['status']})"
        ),
        "description": (
            f"Privacy-masked video from the {study['study_name']}. "
            f"Original data: {study['original_authors']}. "
            f"Masking: {study['masking_method']} via {study['masking_tool']}. "
            f"Reuse context: {study['reuse_context']}."
        ),
        "creator":       [{"@type": "Person", "name": study["archived_by"]}],
        "datePublished": sidecar["archived_at"][:10],
        "license":       "https://creativecommons.org/licenses/by-nc/4.0/",
        "encodingFormat":"video/mp4",
        "contentSize":   f"{sidecar['video']['size_bytes']} B",
        "duration":      f"PT{sidecar['video']['duration_s']:.0f}S",
        "sha256":        sidecar["sha256"],
        "keywords": [
            "behavioral research", "video anonymization",
            "privacy-preserving", "triadic design", "multimodal"
        ],
        "funder":     {"@type": "Organization", "name": "INSERT_FUNDER"},
        "isPartOf":   {"@type": "Dataset", "identifier": "INSERT_COLLECTION_DOI"},
    }

# Demo
dc_meta = make_datacite_metadata(sidecar)
print(json.dumps(dc_meta, indent=2))

## Part 9: BagIt packaging for DANS deposit

DANS Data Station accepts deposits as **BagIt bags** — a folder structure with  
a manifest of checksums that guarantees file integrity across transfer and storage.

```
deposit_root/
  bagit.txt                  ← BagIt version declaration
  bag-info.txt               ← origin, date, organisation
  manifest-sha256.txt        ← one SHA-256 + path per data file
  data/
    *.mp4                    ← masked video files
    *.json                   ← per-file sidecar metadata
    datacite.json            ← DataCite 4.x metadata for DOI registration
    README.txt               ← human-readable deposit description
```

BagIt spec: RFC 8493 — [rfc-editor.org/rfc/rfc8493](https://www.rfc-editor.org/rfc/rfc8493)

In [ ]:
def create_bagit_deposit(archive_group_dir: Path, deposit_root: Path,
                          datacite_meta: dict = None) -> Path:
    """
    Package one group's archived files into a BagIt bag for DANS deposit.
    Zip the returned folder and upload to DANS Data Station SSH.
    """
    bag_dir  = deposit_root / archive_group_dir.name
    data_dir = bag_dir / "data"
    data_dir.mkdir(parents=True, exist_ok=True)

    (bag_dir / "bagit.txt").write_text(
        "BagIt-Version: 1.0\nTag-File-Character-Encoding: UTF-8\n"
    )
    (bag_dir / "bag-info.txt").write_text(
        f"Bagging-Date: {datetime.now(timezone.utc).date().isoformat()}\n"
        "Bag-Software-Agent: thursday-masking summerschool notebook\n"
        "Source-Organization: Radboud University / SYNAPSIS\n"
        "External-Description: Privacy-masked behavioral video — DANS SSH deposit\n"
    )

    checksums = []
    for src in sorted(archive_group_dir.iterdir()):
        if src.suffix in (".mp4", ".wav", ".json", ".csv"):
            dst = data_dir / src.name
            shutil.copy2(src, dst)
            checksums.append((sha256(dst), f"data/{src.name}"))

    if datacite_meta:
        dc_path = data_dir / "datacite.json"
        dc_path.write_text(json.dumps(datacite_meta, indent=2))
        checksums.append((sha256(dc_path), "data/datacite.json"))

    readme = data_dir / "README.txt"
    readme.write_text(
        "ENVISION TRIADIC DESIGN DATASET — MASKED VIDEO DEPOSIT\n"
        "=======================================================\n\n"
        "Contents:\n"
        "  *.mp4          Privacy-masked video (body silhouette, identifiers removed)\n"
        "  *.json         Per-file sidecar: SHA-256, masking parameters, provenance\n"
        "  datacite.json  DataCite 4.x metadata for DOI registration\n\n"
        "Masking procedure:\n"
        "  Tool: MaskAnyone (SYNAPSIS, Radboud University)\n"
        "  Method: YOLO + SAM2 instance segmentation, body silhouette\n"
        "  Preserved: background, table, shared artefacts, timecodes\n\n"
        "Original dataset:\n"
        "  Edelman, J.A. (2011). Understanding Radical Breaks.\n"
        "  PhD dissertation, Stanford University.\n"
        "  License: CC-BY-NC 3.0 US\n\n"
        "For questions: babajide.owoyele@ru.nl\n"
    )
    checksums.append((sha256(readme), "data/README.txt"))

    (bag_dir / "manifest-sha256.txt").write_text(
        "\n".join(f"{chk}  {fname}" for chk, fname in checksums) + "\n"
    )

    print(f"Bag created: {bag_dir}  ({len(checksums)} data files)")
    print("Next: zip the folder and upload to ssh.datastations.nl")
    return bag_dir

print("create_bagit_deposit() ready.")

## Part 10: DANS deposit checklist

Before submitting to DANS Data Station SSH, verify:

- [ ] **Ethics clearance** — IRB/METC protocol number in every sidecar JSON
- [ ] **Consent scope** — participants consented to archived anonymised video
- [ ] **Masking QA** — MaskBench score above threshold (see `03_maskbench_evaluation.ipynb`)
- [ ] **Audio decision** — stripped from open tier, archived separately under restricted access
- [ ] **Checksums verified** — `manifest-sha256.txt` hashes match actual files
- [ ] **DataCite metadata complete** — no `INSERT_` placeholders remain
- [ ] **License alignment** — CC-BY-NC 4.0 consistent with Edelman (2011) CC-BY-NC 3.0
- [ ] **README.txt** — describes masking procedure, original dataset, contact
- [ ] **NARCIS** — link deposit DOI to NARCIS for visibility in Dutch research registry
- [ ] **RDR staging** — if using Radboud Data Repository as pre-DANS staging area

### DANS Data Station SSH deposit steps (manual)

1. Log in at **ssh.datastations.nl** with your SURFconext / institutional account
2. Create a new dataset in your collection (or an existing one)
3. Fill in the DataCite metadata form — or import `datacite.json` directly
4. Upload the BagIt `.zip`
5. Set embargo end date and access tier (open / restricted)
6. Submit for review → receive DOI
7. Add the DOI to your paper's data availability statement

### Programmatic deposit (SWORD v2 API)

DANS Data Station supports SWORD v2 for scripted, reproducible deposits:

```python
import requests

DANS_SWORD_URL = "https://ssh.datastations.nl/sword2/collection/1"
DANS_API_TOKEN = "YOUR_API_TOKEN"   # from ssh.datastations.nl → API Token

headers = {
    "Content-Type":        "application/zip",
    "Content-Disposition": "attachment; filename=envision_group_07_deposit.zip",
    "Packaging":           "http://purl.org/net/sword/package/SimpleZip",
    "Authorization":       f"dataverse_token {DANS_API_TOKEN}",
}
with open("envision_group_07_deposit.zip", "rb") as f:
    response = requests.post(DANS_SWORD_URL, headers=headers, data=f)
print(response.status_code, response.text)
```

See DANS documentation at **dans.knaw.nl** for full API reference.

## Summary: archiving pipeline

```
MaskAnyone output
  group_07__cam_p1__masked.mp4
  group_07__cam_p2__masked.mp4
  group_07__cam_p3__masked.mp4
        │
        ▼  archive_batch()
data/archived/group_07/
  envision__group_07__cam_p1__v01__masked.mp4    ← canonical name
  envision__group_07__cam_p1__v01__masked.json   ← sidecar (SHA256, params)
  envision__group_07__cam_p2__v01__masked.mp4
  envision__group_07__cam_p2__v01__masked.json
  ...
        │
        ▼  write_master_manifest()
data/archived/ARCHIVE_MANIFEST.csv   ← one row per file, all groups
        │
        ▼  extract_audio() / strip_audio()
data/archived/group_07/audio/
  group_07__cam_p1.wav               ← restricted tier
  envision__group_07__cam_p1__v01__masked__silent.mp4  ← open tier
```

**Connection to the rest of Thursday:**  
This session sits at the end of the day (16:15–16:45 or after wrap-up) and answers  
the question: *"I've masked my data — now what do I do with it?"*
